### Allscripts Sunrise (SCM) Drug Exposure Hydration

Runnable-only version.

Notes:
- Reset is active at the top for clean reruns.
- Primary grain stays `dbo_cv3order` + `dbo_cv3medicationextension` (Exponent SCM bronze).
- **Concept codes (RxNorm / NDC)** are merged with the client `medications_scm` pattern: `oto` from performed `dbo_cv3ordertaskoccurrence` + `dbo_sxammordertaskoccurrenceadmin` (`_bronze.prod01_uat`), and **acute-care NDC** from `dbo_scaorder` + `dbo_scamedication` + ranked `dbo_scamedicationndcdim` (`_bronze.sca_acutecare_uat`). SCM `gi` / `pp` values are still used when present.
- OMOP joins use `COALESCE` across those sources (RxNorm: `gi` then `oto`; NDC: `pp` then `oto` then acute-care NDC), then the existing row-level standard/source winner logic.
- Route concepts still use `domain_source_to_concept` when available; unmapped routes stay `0`.


In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_allscripts.drug_exposure;

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_scm.drug_exposure;


In [0]:
%sql
-- DELETE FROM _exponent.omop_silver.drug_exposure
-- WHERE source_system = 'allscripts_scm';


In [0]:
%sql
-- DELETE FROM _exponent.omop_mapping.source_to_drug_exposure
-- WHERE source_system = 'allscripts_scm';


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_drug_exposure AS
WITH sca_med_ndc AS (
  SELECT
    o.OrderGUID,
    ndc.NDC AS sca_ndc_raw,
    ROW_NUMBER() OVER (
      PARTITION BY o.OrderGUID
      ORDER BY m.MedicationID DESC
    ) AS rn
  FROM _bronze.sca_acutecare_uat.dbo_scaorder o
  LEFT JOIN _bronze.sca_acutecare_uat.dbo_scamedication m
    ON m.visitid = o.visitid
   AND m.orderid = o.orderid
   AND m.isactive = 1
  LEFT JOIN (
    SELECT
      ndc_inner.MedicationDimID,
      ndc_inner.NDC,
      ROW_NUMBER() OVER (
        PARTITION BY ndc_inner.MedicationDimID
        ORDER BY ndc_inner.MedicationNDCDimID DESC
      ) AS ndc_rn
    FROM _bronze.sca_acutecare_uat.dbo_scamedicationndcdim ndc_inner
  ) ndc
    ON ndc.MedicationDimID = m.MedicationDimID
   AND ndc.ndc_rn = 1
  WHERE o.isactive = 1
),

sca_med_ndc_1 AS (
  SELECT
    OrderGUID,
    sca_ndc_raw
  FROM sca_med_ndc
  WHERE rn = 1
),

medext_1 AS (
  SELECT *
  FROM (
    SELECT
      medext.*,
      ROW_NUMBER() OVER (
        PARTITION BY medext.GUID
        ORDER BY
          medext.Active DESC,
          medext.TouchedWhen DESC,
          medext.CreatedWhen DESC
      ) AS rn
    FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
    WHERE medext.Active = TRUE
  )
  WHERE rn = 1
),

product_1 AS (
  SELECT *
  FROM (
    SELECT
      p.*,
      ROW_NUMBER() OVER (
        PARTITION BY p.GenericItemID
        ORDER BY
          p.Active DESC,
          p.TouchedWhen DESC,
          p.CreatedWhen DESC,
          p.ProductID DESC
      ) AS rn
    FROM _exponent._bronze_allscripts_scm_prod_01.dbo_sxammproduct p
    WHERE p.Active = TRUE
  )
  WHERE rn = 1
),

product_package_1 AS (
  SELECT *
  FROM (
    SELECT
      pp.*,
      ROW_NUMBER() OVER (
        PARTITION BY pp.ProductID
        ORDER BY
          pp.Active DESC,
          pp.IsRepackaged ASC,
          pp.CreatedWhen DESC,
          pp.TouchedWhen DESC,
          pp.ProductPackageID DESC,
          pp.NDCCode DESC
      ) AS rn
    FROM _exponent._bronze_allscripts_scm_prod_01.dbo_sxammproductpackage pp
    WHERE pp.Active = TRUE
  )
  WHERE rn = 1
),

spp AS (
  SELECT
    p.NDCCode,
    p.ProductID,
    ROW_NUMBER() OVER (
      PARTITION BY p.ProductID
      ORDER BY
        p.Active DESC,
        p.IsRepackaged ASC,
        p.CreatedWhen DESC,
        p.TouchedWhen DESC,
        p.ProductPackageID DESC,
        p.NDCCode DESC
    ) AS rn
  FROM _bronze.prod01_uat.dbo_sxammproductpackage p
),

oto AS (
  SELECT
    ord.GUID AS order_guid,
    p.NDCCode AS oto_ndc,
    otoa.RXCUI_CD AS oto_rxcui,
    ROW_NUMBER() OVER (
      PARTITION BY ord.GUID
      ORDER BY otoa.IsLVP ASC, p.rn ASC, p.NDCCode ASC
    ) AS rn
  FROM _bronze.prod01_uat.dbo_cv3ordertaskoccurrence task
  INNER JOIN _bronze.prod01_uat.dbo_cv3ordertask ot
    ON task.OrderTaskGUID = ot.GUID
  INNER JOIN _bronze.prod01_uat.dbo_cv3order ord
    ON ord.GUID = ot.OrderGUID
  INNER JOIN _bronze.prod01_uat.dbo_cv3medicationextension me
    ON ord.GUID = me.GUID
  INNER JOIN (
    SELECT
      otoa.OrderTaskOccurrenceGUID,
      otoa.ProductID,
      genItem.RxNormCode AS RXCUI_CD,
      genItem.IsLVP,
      ROW_NUMBER() OVER (
        PARTITION BY otoa.OrderTaskOccurrenceGUID
        ORDER BY
          genItem.IsLVP ASC,
          genItem.TouchedWhen DESC,
          otoa.ProductID ASC,
          genItem.RxNormCode ASC
      ) AS ranker
    FROM _bronze.prod01_uat.dbo_sxammordertaskoccurrenceadmin otoa
    JOIN _bronze.prod01_uat.dbo_sxammproduct pr
      ON pr.ProductID = otoa.ProductID
    JOIN _bronze.prod01_uat.dbo_sxammgenericitem genItem
      ON pr.genericitemid = genItem.genericitemid
  ) otoa
    ON otoa.ranker = 1
   AND otoa.OrderTaskOccurrenceGUID = task.GUID
  LEFT JOIN spp p
    ON otoa.ProductID = p.ProductID
   AND p.rn = 1
  WHERE task.TaskStatusCode = 'Performed'
),

oto_1 AS (
  SELECT
    order_guid,
    oto_ndc,
    oto_rxcui
  FROM oto
  WHERE rn = 1
),

rxnorm_candidate_mapping AS (
  SELECT
    source_concept.concept_code,
    source_concept.concept_id AS drug_source_concept_id,
    CASE
      WHEN source_concept.standard_concept = 'S'
       AND source_concept.domain_id = 'Drug'
        THEN source_concept.concept_id
      ELSE standard_concept.concept_id
    END AS drug_concept_id,
    CASE
      WHEN source_concept.standard_concept = 'S'
       AND source_concept.domain_id = 'Drug'
        THEN source_concept.concept_class_id
      ELSE standard_concept.concept_class_id
    END AS drug_concept_class_id,
    CASE
      WHEN source_concept.standard_concept = 'S'
       AND source_concept.domain_id = 'Drug'
        THEN 0
      ELSE 1
    END AS standard_self_priority
  FROM _exponent.omop.concept source_concept
  LEFT JOIN _exponent.omop.concept_relationship concept_relationship
    ON concept_relationship.concept_id_1 = source_concept.concept_id
   AND concept_relationship.relationship_id = 'Maps to'
   AND concept_relationship.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept standard_concept
    ON standard_concept.concept_id = concept_relationship.concept_id_2
   AND standard_concept.standard_concept = 'S'
   AND standard_concept.domain_id = 'Drug'
   AND standard_concept.invalid_reason IS NULL
  WHERE source_concept.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
    AND source_concept.domain_id = 'Drug'
    AND source_concept.invalid_reason IS NULL
),

rxnorm_concept_mapping AS (
  SELECT
    concept_code,
    MIN(drug_source_concept_id) AS drug_source_concept_id,
    MIN_BY(
      drug_concept_id,
      STRUCT(
        standard_self_priority,
        CASE
          WHEN drug_concept_class_id = 'Clinical Drug' THEN 1
          WHEN drug_concept_class_id = 'Branded Drug' THEN 2
          WHEN drug_concept_class_id = 'Clinical Drug Form' THEN 3
          WHEN drug_concept_class_id = 'Ingredient' THEN 4
          ELSE 9
        END,
        drug_concept_id
      )
    ) AS drug_concept_id
  FROM rxnorm_candidate_mapping
  WHERE drug_concept_id IS NOT NULL
  GROUP BY concept_code
),

ndc_candidate_mapping AS (
  SELECT
    source_concept.concept_code,
    source_concept.concept_id AS drug_source_concept_id,
    CASE
      WHEN source_concept.standard_concept = 'S'
       AND source_concept.domain_id = 'Drug'
        THEN source_concept.concept_id
      ELSE standard_concept.concept_id
    END AS drug_concept_id,
    CASE
      WHEN source_concept.standard_concept = 'S'
       AND source_concept.domain_id = 'Drug'
        THEN source_concept.concept_class_id
      ELSE standard_concept.concept_class_id
    END AS drug_concept_class_id,
    CASE
      WHEN source_concept.standard_concept = 'S'
       AND source_concept.domain_id = 'Drug'
        THEN 0
      ELSE 1
    END AS standard_self_priority
  FROM _exponent.omop.concept source_concept
  LEFT JOIN _exponent.omop.concept_relationship concept_relationship
    ON concept_relationship.concept_id_1 = source_concept.concept_id
   AND concept_relationship.relationship_id = 'Maps to'
   AND concept_relationship.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept standard_concept
    ON standard_concept.concept_id = concept_relationship.concept_id_2
   AND standard_concept.standard_concept = 'S'
   AND standard_concept.domain_id = 'Drug'
   AND standard_concept.invalid_reason IS NULL
  WHERE source_concept.vocabulary_id = 'NDC'
    AND source_concept.domain_id = 'Drug'
    AND source_concept.invalid_reason IS NULL
),

ndc_concept_mapping AS (
  SELECT
    concept_code,
    MIN(drug_source_concept_id) AS drug_source_concept_id,
    MIN_BY(
      drug_concept_id,
      STRUCT(
        standard_self_priority,
        CASE
          WHEN drug_concept_class_id = 'Clinical Drug' THEN 1
          WHEN drug_concept_class_id = 'Branded Drug' THEN 2
          WHEN drug_concept_class_id = 'Clinical Drug Form' THEN 3
          WHEN drug_concept_class_id = 'Ingredient' THEN 4
          ELSE 9
        END,
        drug_concept_id
      )
    ) AS drug_concept_id
  FROM ndc_candidate_mapping
  WHERE drug_concept_id IS NOT NULL
  GROUP BY concept_code
)

SELECT
  COALESCE(
    rxnorm_concept_mapping.drug_concept_id,
    ndc_concept_mapping.drug_concept_id,
    0
  ) AS drug_concept_id,

  DATE(COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen)) AS drug_exposure_start_date,
  COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) AS drug_exposure_start_datetime,

  CASE
    WHEN ord.StopDtm IS NOT NULL
     AND ord.StopDtm >= COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen)
    THEN DATE(ord.StopDtm)
    ELSE DATE(COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen))
  END AS drug_exposure_end_date,

  CASE
    WHEN ord.StopDtm IS NOT NULL
     AND ord.StopDtm >= COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen)
    THEN ord.StopDtm
    ELSE COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen)
  END AS drug_exposure_end_datetime,
  CASE
    WHEN ord.StopDtm IS NOT NULL
     AND ord.StopDtm >= COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen)
    THEN DATE(ord.StopDtm)
    ELSE NULL
  END AS verbatim_end_date,
  32817 AS drug_type_concept_id,
  NULL AS stop_reason,
  medext.NumRefills AS refills,
  ROUND(TRY_CAST(medext.DispenseAmount AS DOUBLE), 4) AS quantity,
  NULL AS days_supply,
  NULLIF(TRIM(medext.RxInstructions), '') AS sig,
  COALESCE(route_concept.omop_concept_id, 0) AS route_concept_id,
  NULL AS lot_number,

  NULLIF(
    TRIM(
      COALESCE(
        medext.OrderedAsDisplay,
        medext.OrderedAs,
        gi.GenericItemName,
        p.BrandName,
        ord.Name
      )
    ),
    ''
  ) AS drug_source_value,

  COALESCE(
    rxnorm_concept_mapping.drug_source_concept_id,
    ndc_concept_mapping.drug_source_concept_id,
    0
  ) AS drug_source_concept_id,

  NULLIF(TRIM(medext.OrderRouteCode), '') AS route_source_value,
  NULLIF(TRIM(CAST(COALESCE(medext.DosageLow, medext.Uom) AS STRING)), '') AS dose_unit_source_value,

  CONCAT_WS(
    CHR(31),
    'allscripts_scm',
    'cv3client',
    'GUID',
    CAST(ord.ClientGUID AS STRING)
  ) AS person_source_value,

  CASE
    WHEN ord.CareProviderGUID IS NOT NULL
      THEN CONCAT_WS(
             CHR(31),
             'allscripts_scm',
             'cv3careprovider',
             'GUID',
             CAST(ord.CareProviderGUID AS STRING)
           )
    ELSE NULL
  END AS provider_source_value,

  CASE
    WHEN ord.ClientVisitGUID IS NOT NULL
      THEN CONCAT_WS(
             CHR(31),
             'allscripts_scm',
             'dbo_cv3clientvisit',
             'GUID',
             CAST(ord.ClientVisitGUID AS STRING)
           )
    ELSE NULL
  END AS visit_occurrence_source_value,

  NULL AS visit_detail_source_value,

  CONCAT_WS(
    CHR(31),
    'allscripts_scm',
    'cv3order',
    'GUID',
    CAST(ord.GUID AS STRING)
  ) AS drug_exposure_source_value,

  'allscripts_scm' AS source_system

FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord

INNER JOIN medext_1 medext
  ON medext.GUID = ord.GUID

LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON CAST(gi.DrugID AS STRING) = CAST(medext.PrescriptionGenericItemID AS STRING)
 AND gi.Active = TRUE

LEFT JOIN product_1 p
  ON p.GenericItemID = gi.GenericItemID

LEFT JOIN product_package_1 pp
  ON pp.ProductID = p.ProductID

INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(
       CHR(31),
       'allscripts_scm',
       'cv3client',
       'GUID',
       CAST(ord.ClientGUID AS STRING)
     )
 AND stp.active_flag = TRUE

LEFT JOIN sca_med_ndc_1 sca_ndc
  ON sca_ndc.OrderGUID = ord.GUID

LEFT JOIN oto_1 oto_enrich
  ON oto_enrich.order_guid = ord.GUID

LEFT JOIN rxnorm_concept_mapping
  ON rxnorm_concept_mapping.concept_code = COALESCE(
       NULLIF(TRIM(CAST(gi.RxNormCode AS STRING)), ''),
       NULLIF(TRIM(CAST(oto_enrich.oto_rxcui AS STRING)), '')
     )

LEFT JOIN ndc_concept_mapping
  ON ndc_concept_mapping.concept_code = COALESCE(
       NULLIF(REGEXP_REPLACE(TRIM(CAST(pp.NDCCode AS STRING)), '[^0-9]', ''), ''),
       NULLIF(REGEXP_REPLACE(TRIM(CAST(oto_enrich.oto_ndc AS STRING)), '[^0-9]', ''), ''),
       NULLIF(REGEXP_REPLACE(TRIM(CAST(sca_ndc.sca_ndc_raw AS STRING)), '[^0-9]', ''), '')
     )

LEFT JOIN _exponent.omop_mapping.domain_source_to_concept route_concept
  ON route_concept.source_id = medext.OrderRouteCode
 AND route_concept.domain_id = 'Route'
 AND route_concept.source_system = 'allscripts_scm'
 AND route_concept.active_flag = TRUE

WHERE ord.Active = TRUE
  AND ord.TypeCode = 'Medication'
  AND ord.ClientGUID IS NOT NULL
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) >= TIMESTAMP('1900-01-01')
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) <= CURRENT_TIMESTAMP()
  AND (
       ord.StopDtm IS NULL
       OR (
            ord.StopDtm >= TIMESTAMP('1900-01-01')
        AND ord.StopDtm <= CURRENT_TIMESTAMP()
       )
  );

In [0]:
%sql
MERGE INTO _exponent.omop_silver.drug_exposure AS t
USING silver_drug_exposure AS s
ON t.drug_exposure_source_value = s.drug_exposure_source_value

WHEN MATCHED AND (
     NOT (t.drug_concept_id <=> s.drug_concept_id)
  OR NOT (t.drug_exposure_start_date <=> s.drug_exposure_start_date)
  OR NOT (t.drug_exposure_start_datetime <=> s.drug_exposure_start_datetime)
  OR NOT (t.drug_exposure_end_date <=> s.drug_exposure_end_date)
  OR NOT (t.drug_exposure_end_datetime <=> s.drug_exposure_end_datetime)
  OR NOT (t.verbatim_end_date <=> s.verbatim_end_date)
  OR NOT (t.drug_type_concept_id <=> s.drug_type_concept_id)
  OR NOT (t.stop_reason <=> s.stop_reason)
  OR NOT (t.refills <=> s.refills)
  OR NOT (t.quantity <=> s.quantity)
  OR NOT (t.days_supply <=> s.days_supply)
  OR NOT (t.sig <=> s.sig)
  OR NOT (t.route_concept_id <=> s.route_concept_id)
  OR NOT (t.lot_number <=> s.lot_number)
  OR NOT (t.drug_source_value <=> s.drug_source_value)
  OR NOT (t.drug_source_concept_id <=> s.drug_source_concept_id)
  OR NOT (t.route_source_value <=> s.route_source_value)
  OR NOT (t.dose_unit_source_value <=> s.dose_unit_source_value)
  OR NOT (t.person_source_value <=> s.person_source_value)
  OR NOT (t.provider_source_value <=> s.provider_source_value)
  OR NOT (t.visit_occurrence_source_value <=> s.visit_occurrence_source_value)
  OR NOT (t.visit_detail_source_value <=> s.visit_detail_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.drug_concept_id = s.drug_concept_id,
  t.drug_exposure_start_date = s.drug_exposure_start_date,
  t.drug_exposure_start_datetime = s.drug_exposure_start_datetime,
  t.drug_exposure_end_date = s.drug_exposure_end_date,
  t.drug_exposure_end_datetime = s.drug_exposure_end_datetime,
  t.verbatim_end_date = s.verbatim_end_date,
  t.drug_type_concept_id = s.drug_type_concept_id,
  t.stop_reason = s.stop_reason,
  t.refills = s.refills,
  t.quantity = s.quantity,
  t.days_supply = s.days_supply,
  t.sig = s.sig,
  t.route_concept_id = s.route_concept_id,
  t.lot_number = s.lot_number,
  t.drug_source_value = s.drug_source_value,
  t.drug_source_concept_id = s.drug_source_concept_id,
  t.route_source_value = s.route_source_value,
  t.dose_unit_source_value = s.dose_unit_source_value,
  t.person_source_value = s.person_source_value,
  t.provider_source_value = s.provider_source_value,
  t.visit_occurrence_source_value = s.visit_occurrence_source_value,
  t.visit_detail_source_value = s.visit_detail_source_value,
  t.source_system = s.source_system,
  t.last_mod_tsp = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  drug_concept_id,
  drug_exposure_start_date,
  drug_exposure_start_datetime,
  drug_exposure_end_date,
  drug_exposure_end_datetime,
  verbatim_end_date,
  drug_type_concept_id,
  stop_reason,
  refills,
  quantity,
  days_supply,
  sig,
  route_concept_id,
  lot_number,
  drug_source_value,
  drug_source_concept_id,
  route_source_value,
  dose_unit_source_value,
  person_source_value,
  provider_source_value,
  visit_occurrence_source_value,
  visit_detail_source_value,
  drug_exposure_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.drug_concept_id,
  s.drug_exposure_start_date,
  s.drug_exposure_start_datetime,
  s.drug_exposure_end_date,
  s.drug_exposure_end_datetime,
  s.verbatim_end_date,
  s.drug_type_concept_id,
  s.stop_reason,
  s.refills,
  s.quantity,
  s.days_supply,
  s.sig,
  s.route_concept_id,
  s.lot_number,
  s.drug_source_value,
  s.drug_source_concept_id,
  s.route_source_value,
  s.dose_unit_source_value,
  s.person_source_value,
  s.provider_source_value,
  s.visit_occurrence_source_value,
  s.visit_detail_source_value,
  s.drug_exposure_source_value,
  s.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
-- Defensive cleanup for stale SCM rows from prior runs: OMOP requires end date >= start date.
UPDATE _exponent.omop_silver.drug_exposure
SET
  drug_exposure_end_date = drug_exposure_start_date,
  drug_exposure_end_datetime = drug_exposure_start_datetime,
  verbatim_end_date = NULL,
  last_mod_tsp = CURRENT_TIMESTAMP()
WHERE source_system = 'allscripts_scm'
  AND drug_exposure_end_date < drug_exposure_start_date;

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_drug_exposure (
  source_system,
  drug_exposure_source_value,
  active_flag,
  created_tsp,
  last_mod_tsp
)
SELECT
  s.source_system,
  s.drug_exposure_source_value,
  TRUE,
  CURRENT_TIMESTAMP(),
  COALESCE(s.last_mod_tsp, CURRENT_TIMESTAMP())
FROM (
  SELECT DISTINCT source_system, drug_exposure_source_value, last_mod_tsp
  FROM _exponent.omop_silver.drug_exposure
  WHERE source_system = 'allscripts_scm'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_drug_exposure x
  ON s.drug_exposure_source_value = x.drug_exposure_source_value;

In [0]:
%sql
MERGE INTO _exponent.omop_scm.drug_exposure AS gold
USING (
  SELECT
    sde.drug_exposure_id,
    stp.person_id,
    s.drug_concept_id,
    s.drug_exposure_start_date,
    s.drug_exposure_start_datetime,
    s.drug_exposure_end_date,
    s.drug_exposure_end_datetime,
    s.verbatim_end_date,
    s.drug_type_concept_id,
    s.stop_reason,
    s.refills,
    s.quantity,
    s.days_supply,
    s.sig,
    s.route_concept_id,
    s.lot_number,
    NULL AS provider_id,
    source_to_visit_occurrence.visit_occurrence_id AS visit_occurrence_id,
    NULL AS visit_detail_id,
    s.drug_source_value,
    s.drug_source_concept_id,
    s.route_source_value,
    s.dose_unit_source_value
  FROM _exponent.omop_silver.drug_exposure s
  JOIN _exponent.omop_mapping.source_to_drug_exposure sde
    ON sde.drug_exposure_source_value = s.drug_exposure_source_value
   AND sde.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = s.person_source_value
   AND stp.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_visit_occurrence
    ON source_to_visit_occurrence.visit_occurrence_source_value = s.visit_occurrence_source_value
   AND source_to_visit_occurrence.active_flag = TRUE
  WHERE s.source_system = 'allscripts_scm'
) AS src
ON gold.drug_exposure_id = src.drug_exposure_id

WHEN MATCHED AND (
     NOT (gold.person_id <=> src.person_id)
  OR NOT (gold.drug_concept_id <=> src.drug_concept_id)
  OR NOT (gold.drug_exposure_start_date <=> src.drug_exposure_start_date)
  OR NOT (gold.drug_exposure_start_datetime <=> src.drug_exposure_start_datetime)
  OR NOT (gold.drug_exposure_end_date <=> src.drug_exposure_end_date)
  OR NOT (gold.drug_exposure_end_datetime <=> src.drug_exposure_end_datetime)
  OR NOT (gold.verbatim_end_date <=> src.verbatim_end_date)
  OR NOT (gold.drug_type_concept_id <=> src.drug_type_concept_id)
  OR NOT (gold.stop_reason <=> src.stop_reason)
  OR NOT (gold.refills <=> src.refills)
  OR NOT (gold.quantity <=> src.quantity)
  OR NOT (gold.days_supply <=> src.days_supply)
  OR NOT (gold.sig <=> src.sig)
  OR NOT (gold.route_concept_id <=> src.route_concept_id)
  OR NOT (gold.lot_number <=> src.lot_number)
  OR NOT (gold.provider_id <=> src.provider_id)
  OR NOT (gold.visit_occurrence_id <=> src.visit_occurrence_id)
  OR NOT (gold.visit_detail_id <=> src.visit_detail_id)
  OR NOT (gold.drug_source_value <=> src.drug_source_value)
  OR NOT (gold.drug_source_concept_id <=> src.drug_source_concept_id)
  OR NOT (gold.route_source_value <=> src.route_source_value)
  OR NOT (gold.dose_unit_source_value <=> src.dose_unit_source_value)
)
THEN UPDATE SET
  gold.person_id = src.person_id,
  gold.drug_concept_id = src.drug_concept_id,
  gold.drug_exposure_start_date = src.drug_exposure_start_date,
  gold.drug_exposure_start_datetime = src.drug_exposure_start_datetime,
  gold.drug_exposure_end_date = src.drug_exposure_end_date,
  gold.drug_exposure_end_datetime = src.drug_exposure_end_datetime,
  gold.verbatim_end_date = src.verbatim_end_date,
  gold.drug_type_concept_id = src.drug_type_concept_id,
  gold.stop_reason = src.stop_reason,
  gold.refills = src.refills,
  gold.quantity = src.quantity,
  gold.days_supply = src.days_supply,
  gold.sig = src.sig,
  gold.route_concept_id = src.route_concept_id,
  gold.lot_number = src.lot_number,
  gold.provider_id = src.provider_id,
  gold.visit_occurrence_id = src.visit_occurrence_id,
  gold.visit_detail_id = src.visit_detail_id,
  gold.drug_source_value = src.drug_source_value,
  gold.drug_source_concept_id = src.drug_source_concept_id,
  gold.route_source_value = src.route_source_value,
  gold.dose_unit_source_value = src.dose_unit_source_value

WHEN NOT MATCHED THEN INSERT (
  drug_exposure_id,
  person_id,
  drug_concept_id,
  drug_exposure_start_date,
  drug_exposure_start_datetime,
  drug_exposure_end_date,
  drug_exposure_end_datetime,
  verbatim_end_date,
  drug_type_concept_id,
  stop_reason,
  refills,
  quantity,
  days_supply,
  sig,
  route_concept_id,
  lot_number,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  drug_source_value,
  drug_source_concept_id,
  route_source_value,
  dose_unit_source_value
)
VALUES (
  src.drug_exposure_id,
  src.person_id,
  src.drug_concept_id,
  src.drug_exposure_start_date,
  src.drug_exposure_start_datetime,
  src.drug_exposure_end_date,
  src.drug_exposure_end_datetime,
  src.verbatim_end_date,
  src.drug_type_concept_id,
  src.stop_reason,
  src.refills,
  src.quantity,
  src.days_supply,
  src.sig,
  src.route_concept_id,
  src.lot_number,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.drug_source_value,
  src.drug_source_concept_id,
  src.route_source_value,
  src.dose_unit_source_value
);

In [0]:
%sql
MERGE INTO _exponent.omop_allscripts.drug_exposure AS gold
USING (
  SELECT
    sde.drug_exposure_id,
    stp.person_id,
    s.drug_concept_id,
    s.drug_exposure_start_date,
    s.drug_exposure_start_datetime,
    s.drug_exposure_end_date,
    s.drug_exposure_end_datetime,
    s.verbatim_end_date,
    s.drug_type_concept_id,
    s.stop_reason,
    s.refills,
    s.quantity,
    s.days_supply,
    s.sig,
    s.route_concept_id,
    s.lot_number,
    NULL AS provider_id,
    source_to_visit_occurrence.visit_occurrence_id AS visit_occurrence_id,
    NULL AS visit_detail_id,
    s.drug_source_value,
    s.drug_source_concept_id,
    s.route_source_value,
    s.dose_unit_source_value
  FROM _exponent.omop_silver.drug_exposure s
  JOIN _exponent.omop_mapping.source_to_drug_exposure sde
    ON sde.drug_exposure_source_value = s.drug_exposure_source_value
   AND sde.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = s.person_source_value
   AND stp.source_system = 'allscripts_scm'
   AND stp.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_visit_occurrence
    ON source_to_visit_occurrence.visit_occurrence_source_value = s.visit_occurrence_source_value
   AND source_to_visit_occurrence.active_flag = TRUE
  WHERE s.source_system = 'allscripts_scm'
) AS src
ON gold.drug_exposure_id = src.drug_exposure_id

WHEN MATCHED AND (
     NOT (gold.person_id <=> src.person_id)
  OR NOT (gold.drug_concept_id <=> src.drug_concept_id)
  OR NOT (gold.drug_exposure_start_date <=> src.drug_exposure_start_date)
  OR NOT (gold.drug_exposure_start_datetime <=> src.drug_exposure_start_datetime)
  OR NOT (gold.drug_exposure_end_date <=> src.drug_exposure_end_date)
  OR NOT (gold.drug_exposure_end_datetime <=> src.drug_exposure_end_datetime)
  OR NOT (gold.verbatim_end_date <=> src.verbatim_end_date)
  OR NOT (gold.drug_type_concept_id <=> src.drug_type_concept_id)
  OR NOT (gold.stop_reason <=> src.stop_reason)
  OR NOT (gold.refills <=> src.refills)
  OR NOT (gold.quantity <=> src.quantity)
  OR NOT (gold.days_supply <=> src.days_supply)
  OR NOT (gold.sig <=> src.sig)
  OR NOT (gold.route_concept_id <=> src.route_concept_id)
  OR NOT (gold.lot_number <=> src.lot_number)
  OR NOT (gold.provider_id <=> src.provider_id)
  OR NOT (gold.visit_occurrence_id <=> src.visit_occurrence_id)
  OR NOT (gold.visit_detail_id <=> src.visit_detail_id)
  OR NOT (gold.drug_source_value <=> src.drug_source_value)
  OR NOT (gold.drug_source_concept_id <=> src.drug_source_concept_id)
  OR NOT (gold.route_source_value <=> src.route_source_value)
  OR NOT (gold.dose_unit_source_value <=> src.dose_unit_source_value)
)
THEN UPDATE SET
  gold.person_id = src.person_id,
  gold.drug_concept_id = src.drug_concept_id,
  gold.drug_exposure_start_date = src.drug_exposure_start_date,
  gold.drug_exposure_start_datetime = src.drug_exposure_start_datetime,
  gold.drug_exposure_end_date = src.drug_exposure_end_date,
  gold.drug_exposure_end_datetime = src.drug_exposure_end_datetime,
  gold.verbatim_end_date = src.verbatim_end_date,
  gold.drug_type_concept_id = src.drug_type_concept_id,
  gold.stop_reason = src.stop_reason,
  gold.refills = src.refills,
  gold.quantity = src.quantity,
  gold.days_supply = src.days_supply,
  gold.sig = src.sig,
  gold.route_concept_id = src.route_concept_id,
  gold.lot_number = src.lot_number,
  gold.provider_id = src.provider_id,
  gold.visit_occurrence_id = src.visit_occurrence_id,
  gold.visit_detail_id = src.visit_detail_id,
  gold.drug_source_value = src.drug_source_value,
  gold.drug_source_concept_id = src.drug_source_concept_id,
  gold.route_source_value = src.route_source_value,
  gold.dose_unit_source_value = src.dose_unit_source_value

WHEN NOT MATCHED THEN INSERT (
  drug_exposure_id,
  person_id,
  drug_concept_id,
  drug_exposure_start_date,
  drug_exposure_start_datetime,
  drug_exposure_end_date,
  drug_exposure_end_datetime,
  verbatim_end_date,
  drug_type_concept_id,
  stop_reason,
  refills,
  quantity,
  days_supply,
  sig,
  route_concept_id,
  lot_number,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  drug_source_value,
  drug_source_concept_id,
  route_source_value,
  dose_unit_source_value
)
VALUES (
  src.drug_exposure_id,
  src.person_id,
  src.drug_concept_id,
  src.drug_exposure_start_date,
  src.drug_exposure_start_datetime,
  src.drug_exposure_end_date,
  src.drug_exposure_end_datetime,
  src.verbatim_end_date,
  src.drug_type_concept_id,
  src.stop_reason,
  src.refills,
  src.quantity,
  src.days_supply,
  src.sig,
  src.route_concept_id,
  src.lot_number,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.drug_source_value,
  src.drug_source_concept_id,
  src.route_source_value,
  src.dose_unit_source_value
);

In [0]:
%sql
-- Defensive cleanup for stale SCM gold rows from prior runs or partial reloads.
UPDATE _exponent.omop_scm.drug_exposure
SET
  drug_exposure_end_date = drug_exposure_start_date,
  drug_exposure_end_datetime = drug_exposure_start_datetime,
  verbatim_end_date = NULL
WHERE drug_exposure_end_date < drug_exposure_start_date;

In [0]:
%sql
SELECT
  'drug_end_before_start' AS check_name,
  COUNT(*) AS fail_count
FROM _exponent.omop_scm.drug_exposure
WHERE drug_exposure_end_date < drug_exposure_start_date;